In [ ]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# =====================================================
# LOAD DATA
# =====================================================

files = [
    "../data/2018.xlsx",
    "../data/2019.xlsx",
    "../data/2020.xlsx",
    "../data/2021.xlsx",
    "../data/2022.xlsx"
]

dfs = [pd.read_excel(f) for f in files]

df = pd.concat(dfs, ignore_index=True)

print("Original Shape:", df.shape)

# =====================================================
# CLEAN DATA
# =====================================================

df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

df["time"] = pd.to_datetime(df["time"])

df["month"] = df["time"].dt.month
df["day"] = df["time"].dt.day
df["hour"] = df["time"].dt.hour

# =====================================================
# CREATE LAG FEATURES
# =====================================================

df = df.sort_values("time")

df["sst_lag1"] = df["sst"].shift(1)
df["sst_lag2"] = df["sst"].shift(2)
df["sst_lag3"] = df["sst"].shift(3)
df["sst_lag4"] = df["sst"].shift(4)

df = df.dropna(
    subset=[
        "sst_lag1",
        "sst_lag2",
        "sst_lag3",
        "sst_lag4"
    ]
)

print("After Lag Features:", df.shape)

# =====================================================
# FEATURES
# =====================================================

X = df[
    [
        "sst_lag1",
        "sst_lag2",
        "sst_lag3",
        "sst_lag4",
        "month",
        "day",
        "hour"
    ]
]

y = df["sst"]

print("X Shape:", X.shape)
print("y Shape:", y.shape)

# =====================================================
# TIME SERIES SPLIT
# =====================================================

train_size = int(len(df) * 0.8)

X_train = X.iloc[:train_size]
X_test = X.iloc[train_size:]

y_train = y.iloc[:train_size]
y_test = y.iloc[train_size:]

print("Train:", X_train.shape)
print("Test :", X_test.shape)

# =====================================================
# SCALING
# =====================================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# =====================================================
# TENSORS
# =====================================================

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(
    y_train.values,
    dtype=torch.float32
).reshape(-1, 1)

y_test = torch.tensor(
    y_test.values,
    dtype=torch.float32
).reshape(-1, 1)

# =====================================================
# MODEL
# =====================================================

class SSTModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(7, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.network(x)

model = SSTModel()

print(model)

# =====================================================
# LOSS + OPTIMIZER
# =====================================================

loss_fn = nn.MSELoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

# =====================================================
# TRAIN
# =====================================================

epochs = 500

for epoch in range(epochs):

    model.train()

    predictions = model(X_train)

    loss = loss_fn(
        predictions,
        y_train
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(
            f"Epoch {epoch:3d} | Loss = {loss.item():.6f}"
        )

# =====================================================
# EVALUATE
# =====================================================

model.eval()

with torch.no_grad():
    predictions = model(X_test)

pred_np = predictions.numpy()
y_np = y_test.numpy()

mse = mean_squared_error(y_np, pred_np)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_np, pred_np)
r2 = r2_score(y_np, pred_np)

print("\n========================")
print("MODEL PERFORMANCE")
print("========================")

print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R²   : {r2:.4f}")

# =====================================================
# SAMPLE PREDICTIONS
# =====================================================



Original Shape: (14608, 10)
After Lag Features: (14604, 11)
X Shape: (14604, 7)
y Shape: (14604,)
Train: (11683, 7)
Test : (2921, 7)
SSTModel(
  (network): Sequential(
    (0): Linear(in_features=7, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=16, bias=True)
    (5): ReLU()
    (6): Linear(in_features=16, out_features=1, bias=True)
  )
)
Epoch   0 | Loss = 860.286987
Epoch  50 | Loss = 706.348694
Epoch 100 | Loss = 103.986336
Epoch 150 | Loss = 46.596481
Epoch 200 | Loss = 25.810085
Epoch 250 | Loss = 18.287731
Epoch 300 | Loss = 14.541224
Epoch 350 | Loss = 11.746403
Epoch 400 | Loss = 9.257827
Epoch 450 | Loss = 6.925677

MODEL PERFORMANCE
MSE  : 6.0799
RMSE : 2.4657
MAE  : 1.9486
R²   : -7.7710
